# 03: Recommendation Models & Validation Tuning

This notebook fits our recommendation engines (Popularity, Item-CF, User-CF, Association Rules) on the training set, and performs weight tuning for the Hybrid model on the validation set.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.db import query_to_dataframe
from src.recommenders import PopularityRecommender, ItemCFRecommender, UserCFRecommender, AssociationRuleRecommender, HybridRecommender

## 1. Load Data for Training

In [ ]:
query = """
SELECT i.customer_id, ii.stock_code, i.invoice_date
FROM invoices i
JOIN invoice_items ii ON i.invoice_no = ii.invoice_no
WHERE i.is_cancellation = 0
"""
df = query_to_dataframe(query)
df["invoice_date"] = pd.to_datetime(df["invoice_date"])

## 2. Temporal Split (Train / Val)

In [ ]:
train_end = pd.to_datetime("2011-09-01")
val_end = pd.to_datetime("2011-10-31")

train_df = df[df["invoice_date"] < train_end]
val_df = df[(df["invoice_date"] >= train_end) & (df["invoice_date"] < val_end)]
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

## 3. Fit Component Models

In [ ]:
pop_rec = PopularityRecommender()
pop_rec.fit(train_df)

item_rec = ItemCFRecommender()
item_rec.fit(train_df)

user_rec = UserCFRecommender()
user_rec.fit(train_df)

rules_rec = AssociationRuleRecommender(min_support=0.01, min_confidence=0.1)
# Convert format for rules (InvoiceNo x StockCode). Since we only pulled customer_id, we can group by customer_id.
# Actually, let's fetch InvoiceNo for correct basket generation.
query_basket = """
SELECT i.invoice_no, ii.stock_code, i.invoice_date
FROM invoices i
JOIN invoice_items ii ON i.invoice_no = ii.invoice_no
WHERE i.is_cancellation = 0
"""
basket_df = query_to_dataframe(query_basket)
basket_df["invoice_date"] = pd.to_datetime(basket_df["invoice_date"])
train_basket = basket_df[basket_df["invoice_date"] < train_end]

rules_rec.fit(train_basket)
print("Models fitted successfully!")